# kdd_seq_04_contrastive_gru: Contrastive Sequential Representation Learning

Learns view-invariant sequence embeddings via symmetric NT-Xent (InfoNCE) contrastive loss over stochastic subsequence crops (60%–95%) and random token masking (15%). Because positive pairs have different cropped lengths, the encoder is explicitly discouraged from encoding sequence volume.

In [1]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

WORK = "/content/drive/MyDrive/kdd_seq"
sys.path.insert(0, WORK)
import kdd_seq_common as C

# ------------------------------- CONFIG -------------------------------
METHOD = "gru_cl"
MAX_LEN = 128          # most recent 128 tokens (covers >95% of sequences completely)
BATCH_SIZE = 512
EPOCHS = 15
LR = 3e-3
WEIGHT_DECAY = 1e-4
K_GRID = list(range(2, 9))
K_FINAL = 8            # matches summary_markov.json; K=5 also evaluated downstream
DOWNSTREAM_KS = [5, 8] # compare both behavioural regime (K=5) and Markov BIC match (K=8)
MIN_SHARE = 0.03
CV_SAMPLE, CV_SPLITS, CV_REPEATS = 60000, 3, 3
SEEDS_STAB, N_BOOT = 5, 8
SEED = 42
# ----------------------------------------------------------------------
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(f"{WORK}/results", exist_ok=True)

A = C.load_artifacts(WORK)
V = A["vocab"]["V"]; NAMES = A["vocab"]["names"]
F_tr, F_te, y_tr, y_te = A["F_train"], A["F_test"], A["y_train"], A["y_test"]
nev_tr, nev_te = A["base_train"]["n_events"].values, A["base_test"]["n_events"].values
print(f"Device: {DEVICE} | Train: {F_tr.shape[0]} | Test: {F_te.shape[0]} | V={V}")


Mounted at /content/drive
Device: cuda | Train: 120542 | Test: 80362 | V=19


In [2]:
# ==============================================================================
# Shared Sequence Packing, Spherical Clustering & Evaluation Helpers
# ==============================================================================
BOS = V          # 19
EOS = V + 1      # 20
PAD = V + 2      # 21
VOCAB_SIZE = V + 3  # 22

def pack_sequences(tokens, offsets, max_len=128):
    """Packs flat CSR sequences (tokens, offsets) into a dense 2D array (N, L_max+2)
    with BOS at index 0, up to max_len most recent tokens (preserving recency), EOS, and PAD."""
    N = len(offsets) - 1
    lengths = np.diff(offsets)
    L_cap = int(min(max_len, lengths.max()))
    out = np.full((N, L_cap + 2), PAD, dtype=np.int64)
    out[:, 0] = BOS
    eff_lens = np.minimum(lengths, L_cap).astype(np.int64)
    for i in range(N):
        s, e = offsets[i], offsets[i + 1]
        L = e - s
        if L > L_cap:
            s = e - L_cap
            L = L_cap
        out[i, 1:1 + L] = tokens[s:e]
        out[i, 1 + L] = EOS
    return out, eff_lens, lengths.astype(np.int64)


class SphericalClusterer:
    """K-Means on L2-normalized embeddings with descending cluster size ordering
    (cluster 0 = largest) and temperature-scaled cosine softmax posteriors."""
    def __init__(self, K, seed=0, n_init=10, temp=0.15):
        self.K = K
        self.seed = seed
        self.n_init = n_init
        self.temp = temp

    def fit(self, Z):
        km = KMeans(self.K, n_init=self.n_init, random_state=self.seed).fit(Z)
        counts = np.bincount(km.labels_, minlength=self.K)
        order = np.argsort(-counts)
        self.centroids_ = km.cluster_centers_[order]
        norms = np.linalg.norm(self.centroids_, axis=1, keepdims=True)
        self.centroids_norm_ = self.centroids_ / np.maximum(norms, 1e-12)
        self.inertia_ = float(km.inertia_)
        return self

    def cosine_sim(self, Z):
        Zn = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)
        return (Zn @ self.centroids_norm_.T).astype(np.float32)

    def predict_proba(self, Z):
        logits = self.cosine_sim(Z) / self.temp
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_l = np.exp(logits)
        return (exp_l / exp_l.sum(axis=1, keepdims=True)).astype(np.float32)

    def predict(self, Z):
        return self.cosine_sim(Z).argmax(axis=1)


def residualize_volume(Z_tr, Z_te, nev_tr, nev_te, len_tr, len_te):
    """Regresses out log1p(n_events) and log1p(seq_len) from embeddings (fitted on train only),
    then re-normalizes to unit L2 length."""
    X_tr = np.column_stack([np.ones(len(nev_tr)), np.log1p(nev_tr), np.log1p(len_tr)])
    X_te = np.column_stack([np.ones(len(nev_te)), np.log1p(nev_te), np.log1p(len_te)])
    W, _, _, _ = np.linalg.lstsq(X_tr, Z_tr, rcond=None)
    R_tr = Z_tr - X_tr @ W
    R_te = Z_te - X_te @ W
    R_tr /= np.maximum(np.linalg.norm(R_tr, axis=1, keepdims=True), 1e-12)
    R_te /= np.maximum(np.linalg.norm(R_te, axis=1, keepdims=True), 1e-12)
    return R_tr.astype(np.float32), R_te.astype(np.float32)


def describe_clusters_from_F(F, R_soft, V, names, alpha=0.1, cap=50, top=6, min_mass=0.005):
    """Describes any clustering using the empirical transition matrix F[:, V:] in the
    exact same characteristic lift & frequency format as the Markov mixture."""
    K = R_soft.shape[1]
    n = np.asarray(F[:, V:].sum(1)).ravel()
    Fc = (sp.diags(np.minimum(1.0, cap / np.maximum(n, 1))) @ F).tocsr() if cap else F
    S = np.asarray(Fc.T.dot(R_soft)).T[:, V:].reshape(K, V, V)
    G = S.sum(0) + alpha
    Pg = G / G.sum(1, keepdims=True)
    out = {}
    for k in range(K):
        Pk = (S[k] + alpha) / (S[k] + alpha).sum(1, keepdims=True)
        mass = S[k] / np.maximum(S[k].sum(), 1e-12)
        lift = np.where(mass >= min_mass, Pk / Pg, 0)
        li = np.dstack(np.unravel_index(np.argsort(-lift, axis=None)[:top], lift.shape))[0]
        fi = np.dstack(np.unravel_index(np.argsort(-mass, axis=None)[:top], mass.shape))[0]
        out[k] = dict(
            share=float(R_soft[:, k].mean()),
            characteristic=[f"{names[a]} -> {names[b]} (lift {lift[a,b]:.1f})" for a, b in li],
            frequent=[f"{names[a]} -> {names[b]} ({mass[a,b]*100:.1f}%)" for a, b in fi]
        )
    return out


X_tr, eff_tr, len_tr = pack_sequences(A["tok_train"], A["off_train"], MAX_LEN)
X_te, eff_te, len_te = pack_sequences(A["tok_test"],  A["off_test"],  MAX_LEN)
print("Packed train tensor:", X_tr.shape, "| Packed test tensor:", X_te.shape)


Packed train tensor: (120542, 130) | Packed test tensor: (80362, 130)


## 1. Neural Architecture & Unsupervised Pretraining
Trained strictly on `X_tr` without dropout labels (`y_tr`) to prevent any target leakage.

In [3]:
def augment_batch_gpu(x, mask_prob=0.15, crop_min=0.6, unk_idx=18, bos_idx=BOS, eos_idx=EOS, pad_idx=PAD):
    """Fast vectorized GPU sequence augmentations:
    1. Random token masking (15% of active tokens -> <unk>)
    2. Random contiguous subsequence cropping (retaining 60-95% of tokens for sequences with L >= 5)"""
    B, T = x.shape
    out = x.clone()
    device = x.device
    active = (out != pad_idx) & (out != bos_idx) & (out != eos_idx)
    lengths = active.sum(dim=1)
    rand_m = torch.rand((B, T), device=device)
    out[active & (rand_m < mask_prob)] = unk_idx
    crop_ratio = crop_min + (0.95 - crop_min) * torch.rand((B, 1), device=device)
    keep_len = torch.clamp((lengths.unsqueeze(1).float() * crop_ratio).long(), min=1)
    max_start = torch.clamp(lengths.unsqueeze(1) - keep_len, min=0)
    start_idx = (torch.rand((B, 1), device=device) * (max_start.float() + 1.0)).long() + 1
    end_idx = start_idx + keep_len
    pos = torch.arange(T, device=device).unsqueeze(0)
    drop_crop = active & (lengths.unsqueeze(1) >= 5) & ((pos < start_idx) | (pos >= end_idx))
    out[drop_crop] = pad_idx
    return out


class ContrastiveGRU(nn.Module):
    """SimCLR / TS2Vec style Contrastive GRU with InfoNCE (NT-Xent) objective."""
    def __init__(self, vocab_size=VOCAB_SIZE, d_emb=32, d_hid=64, d_proj=32, n_layers=2, dropout=0.1, temp=0.1, pad_idx=PAD):
        super().__init__()
        self.pad_idx = pad_idx
        self.temp = temp
        self.emb = nn.Embedding(vocab_size, d_emb, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(d_emb, d_hid, num_layers=n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.proj = nn.Sequential(nn.Linear(d_hid, d_hid), nn.GELU(), nn.Linear(d_hid, d_proj))

    def forward_backbone(self, x):
        mask = (x != self.pad_idx).unsqueeze(-1).float()
        h, _ = self.gru(self.drop(self.emb(x)))
        mean_h = (h * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return F.normalize(mean_h, p=2, dim=-1)

    def loss(self, x):
        x1 = augment_batch_gpu(x)
        x2 = augment_batch_gpu(x)
        h1, h2 = self.forward_backbone(x1), self.forward_backbone(x2)
        z1 = F.normalize(self.proj(h1), p=2, dim=-1)
        z2 = F.normalize(self.proj(h2), p=2, dim=-1)
        B = z1.size(0)
        z = torch.cat([z1, z2], dim=0)
        sim = torch.matmul(z, z.T) / self.temp
        sim.fill_diagonal_(-1e9)
        targets = torch.cat([torch.arange(B, 2 * B, device=z.device),
                             torch.arange(0, B, device=z.device)], dim=0)
        return F.cross_entropy(sim, targets)

    @torch.no_grad()
    def encode(self, x):
        return self.forward_backbone(x)

MODEL_CLS = ContrastiveGRU

idx_tr, idx_va = train_test_split(np.arange(len(X_tr)), test_size=0.1, random_state=SEED)
train_loader = DataLoader(TensorDataset(torch.tensor(X_tr[idx_tr], dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_tr[idx_va], dtype=torch.long)), batch_size=BATCH_SIZE * 2, shuffle=False)

model = MODEL_CLS().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best_val_loss, best_state = np.inf, None
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss, n_tr = 0.0, 0
    for (bx,) in train_loader:
        bx = bx.to(DEVICE)
        opt.zero_grad()
        loss = model.loss(bx)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_loss += loss.item() * len(bx); n_tr += len(bx)
    sched.step()
    model.eval()
    va_loss, n_va = 0.0, 0
    with torch.no_grad():
        for (bx,) in val_loader:
            bx = bx.to(DEVICE)
            va_loss += model.loss(bx).item() * len(bx); n_va += len(bx)
    tr_loss /= n_tr; va_loss /= n_va
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if ep == 1 or ep % 3 == 0 or ep == EPOCHS:
        print(f"Epoch {ep:02d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} (ppl={np.exp(va_loss):.2f}) | {time.time()-t0:.1f}s")

model.load_state_dict(best_state)
print(f"Best validation loss: {best_val_loss:.4f} (perplexity: {np.exp(best_val_loss):.2f})")


Epoch 01/15 | train_loss=3.5332 | val_loss=3.3627 (ppl=28.87) | 7.3s
Epoch 03/15 | train_loss=2.4135 | val_loss=2.8027 (ppl=16.49) | 5.8s
Epoch 06/15 | train_loss=2.1543 | val_loss=2.5866 (ppl=13.28) | 5.8s
Epoch 09/15 | train_loss=2.0747 | val_loss=2.4969 (ppl=12.14) | 5.9s
Epoch 12/15 | train_loss=2.0313 | val_loss=2.4552 (ppl=11.65) | 6.1s
Epoch 15/15 | train_loss=2.0211 | val_loss=2.4375 (ppl=11.44) | 6.0s
Best validation loss: 2.4332 (perplexity: 11.40)


## 2. Embedding Extraction & Volume Residualization
Extracts L2-normalized embeddings (`Z_tr`, `Z_te`) and computes a volume-orthogonalized copy (`Z_tr_orth`, `Z_te_orth`) by regressing out $\log(1 + n_{\text{events}})$ and $\log(1 + L_{\text{seq}})$.

In [4]:
@torch.no_grad()
def extract_embeddings(model, X_np, batch_size=1024):
    model.eval()
    zs = []
    for i in range(0, len(X_np), batch_size):
        bx = torch.tensor(X_np[i:i + batch_size], dtype=torch.long, device=DEVICE)
        zs.append(model.encode(bx).cpu().numpy())
    return np.vstack(zs).astype(np.float32)

Z_tr = extract_embeddings(model, X_tr)
Z_te = extract_embeddings(model, X_te)
Z_tr_orth, Z_te_orth = residualize_volume(Z_tr, Z_te, nev_tr, nev_te, len_tr, len_te)

pca16 = PCA(n_components=min(16, Z_tr.shape[1]), random_state=0).fit(Z_tr)
Z_tr_pca = pca16.transform(Z_tr).astype(np.float32)
Z_te_pca = pca16.transform(Z_te).astype(np.float32)

pca16_orth = PCA(n_components=min(16, Z_tr_orth.shape[1]), random_state=0).fit(Z_tr_orth)
Z_tr_orth_pca = pca16_orth.transform(Z_tr_orth).astype(np.float32)
Z_te_orth_pca = pca16_orth.transform(Z_te_orth).astype(np.float32)

print("Z_tr:", Z_tr.shape, "| PCA16 explained var: %.3f" % pca16.explained_variance_ratio_.sum())


Z_tr: (120542, 64) | PCA16 explained var: 0.936


## 3. Clustering Grid ($K = 2 \dots 8$): Shared Space, Native Space & Volume Check

In [5]:
rows, models = [], {}
for K in K_GRID:
    t0 = time.time()
    m = SphericalClusterer(K, seed=0, n_init=10).fit(Z_tr)
    models[K] = m
    lab_tr, lab_te = m.predict(Z_tr), m.predict(Z_te)
    P_tr = m.predict_proba(Z_tr)
    r = dict(K=K, inertia=m.inertia_, mean_max_posterior=float(P_tr.max(1).mean()))
    # Shared evaluation space (comparable across all methods including Markov)
    r.update(C.cluster_quality(A["eval_train"], lab_tr))
    # Native embedding space
    nat = C.cluster_quality(Z_tr, lab_tr)
    r["native_silhouette"] = nat["silhouette"]
    r["native_davies_bouldin"] = nat["davies_bouldin"]
    r["native_calinski_harabasz"] = nat["calinski_harabasz"]
    # Volume confounding & dropout association
    r.update(C.volume_agreement(lab_tr, nev_tr))
    r["cramers_v_train"] = C.cramers_v(lab_tr, y_tr)
    r["cramers_v_test"]  = C.cramers_v(lab_te, y_te)
    rows.append(r)
    print(f"K={K} done in {time.time()-t0:.1f}s | shared Sil={r['silhouette']:.4f} | native Sil={r['native_silhouette']:.4f} | ARI_vol={r['ari_vs_volume']:.4f}")

grid = pd.DataFrame(rows).set_index("K")
grid.round(4).to_csv(f"{WORK}/results/{METHOD}_grid.csv")
display(grid.round(4))


K=2 done in 16.2s | shared Sil=0.1820 | native Sil=0.3201 | ARI_vol=0.5506
K=3 done in 16.6s | shared Sil=0.2152 | native Sil=0.3902 | ARI_vol=0.6957
K=4 done in 17.7s | shared Sil=0.2063 | native Sil=0.4180 | ARI_vol=0.5942
K=5 done in 17.7s | shared Sil=0.1878 | native Sil=0.4060 | ARI_vol=0.6438
K=6 done in 19.2s | shared Sil=0.1935 | native Sil=0.3945 | ARI_vol=0.5665
K=7 done in 24.3s | shared Sil=0.1894 | native Sil=0.3915 | ARI_vol=0.5878
K=8 done in 20.9s | shared Sil=0.1899 | native Sil=0.3848 | ARI_vol=0.4987


,inertia,mean_max_posterior,n_clusters,min_cluster_share,silhouette,davies_bouldin,calinski_harabasz,native_silhouette,native_davies_bouldin,native_calinski_harabasz,ari_vs_volume,nmi_vs_volume,eta2_logvolume,cramers_v_train,cramers_v_test
K,,,,,,,,,,,,,,,
2,77504.4297,0.9545,2,0.4377,0.1820,2.4971,2774.7383,0.3201,1.4661,8394.4043,0.5506,0.4931,0.5808,0.2895,0.2868
3,55138.5859,0.9419,3,0.2862,0.2152,2.6695,2734.1248,0.3902,1.2331,10080.8408,0.6957,0.6726,0.8196,0.4194,0.4179
4,42449.8672,0.9243,4,0.2076,0.2063,3.5797,2209.4065,0.4180,1.1503,10808.7178,0.5942,0.6646,0.8962,0.4724,0.4685
5,36469.7930,0.8867,5,0.1693,0.1878,4.8664,1712.6860,0.4060,1.2747,10247.4014,0.6438,0.6906,0.9265,0.4927,0.4860
6,33098.3398,0.8396,6,0.1224,0.1935,6.1981,1418.9492,0.3945,1.3096,9427.5664,0.5665,0.6670,0.9370,0.4953,0.4878
7,30802.8008,0.8123,7,0.0743,0.1894,5.7781,1278.0741,0.3915,1.4192,8704.0322,0.5878,0.6480,0.9379,0.4947,0.4892
8,28979.0723,0.7671,8,0.0585,0.1899,5.7591,1142.2637,0.3848,1.4255,8134.6514,0.4987,0.6295,0.9423,0.4983,0.4916


## 4. Cluster Profiles & Behavioural Interpretation (Characteristic Transitions)

In [6]:
for K_view in DOWNSTREAM_KS:
    print(f"\n=================== {METHOD.upper()} PROFILE (K={K_view}) ===================")
    mk = models[K_view]
    l_tr = mk.predict(Z_tr)
    print(C.cluster_profile(l_tr, y_tr, nev_tr))
    desc = describe_clusters_from_F(F_tr, mk.predict_proba(Z_tr), V, NAMES)
    json.dump(desc, open(f"{WORK}/results/{METHOD}_cluster_descriptions_K{K_view}.json", "w"), indent=1)
    for k, d in desc.items():
        print(f"\n--- Cluster {k} (soft share {d['share']:.3f}) ---")
        print("  characteristic:", "; ".join(d["characteristic"][:4]))
        print("  frequent:      ", "; ".join(d["frequent"][:4]))



=================== GRU_CL PROFILE (K=5) ===================
    size  share  dropout  mean_events  median_events
c                                                   
0  31004  0.257    0.951        2.197            1.0
1  23671  0.196    0.884       19.045           17.0
2  23056  0.191    0.745       50.335           47.0
3  22405  0.186    0.403      185.614          142.0
4  20406  0.169    0.929        8.155            7.0

--- Cluster 0 (soft share 0.255) ---
  characteristic: server_navigate_unknown -> <gap_48h+> (lift 3.7); server_navigate_unknown -> <gap_0.5-6h> (lift 3.2); browser_video_video -> browser_access_sequential (lift 3.0); server_access_sequential -> browser_access_sequential (lift 2.6)
  frequent:       server_navigate_unknown -> server_access_unknown (8.4%); server_access_sequential -> browser_page_close_unknown (6.4%); server_access_unknown -> server_access_sequential (5.1%); browser_problem_problem -> server_problem_problem (5.0%)

--- Cluster 1 (soft share 0.1

## 5. Stability Analysis (Seed, Bootstrap & Train $\rightarrow$ Test Transfer)

In [7]:
stab_by_k = {}
for K_eval in DOWNSTREAM_KS:
    mk = models[K_eval]
    lab_tr_k, lab_te_k = mk.predict(Z_tr), mk.predict(Z_te)
    seed_labels = [SphericalClusterer(K_eval, seed=100 + s, n_init=5).fit(Z_tr).predict(Z_tr) for s in range(SEEDS_STAB)]
    st = C.seed_stability(seed_labels)
    st.update(C.bootstrap_stability(
        fit=lambda idx, seed: SphericalClusterer(K_eval, seed=seed, n_init=5).fit(Z_tr[idx]),
        predict=lambda cm, rows: cm.predict(Z_tr[rows]),
        n_rows=Z_tr.shape[0], ref_labels=lab_tr_k, n_boot=N_BOOT))
    st.update(C.transfer_consistency(lab_tr_k, y_tr, lab_te_k, y_te, K_eval))
    stab_by_k[K_eval] = {k: round(v, 4) for k, v in st.items()}
    print(f"\n=== Stability ({METHOD}, K={K_eval}) ===")
    print(pd.Series(stab_by_k[K_eval]).to_string())



=== Stability (gru_cl, K=5) ===
seed_ari_mean                 1.0000
seed_ari_min                  1.0000
boot_ari_mean                 0.9948
boot_ari_std                  0.0024
share_L1_train_vs_test        0.0033
dropout_corr_train_vs_test    0.9998
cramers_v_train               0.4927
cramers_v_test                0.4860

=== Stability (gru_cl, K=8) ===
seed_ari_mean                 0.8719
seed_ari_min                  0.7771
boot_ari_mean                 0.8474
boot_ari_std                  0.1201
share_L1_train_vs_test        0.0077
dropout_corr_train_vs_test    0.9999
cramers_v_train               0.4983
cramers_v_test                0.4916


## 6. Downstream Dropout Prediction (Train CV & Real Test Set)
Evaluates `hard`, `soft`, `hard+soft`, `cos_sim` (analogous to Markov `avgll`), continuous neural embeddings (`emb_pca16`), volume-orthogonalized embeddings (`emb_vol_orth`), and `null_random`.

In [8]:
def make_variants(model_k, K, Z_a, Z_pca_a, Z_orth_pca_a, seed=0):
    P = model_k.predict_proba(Z_a)
    lab = P.argmax(1)
    cs = model_k.cosine_sim(Z_a)
    rng = np.random.default_rng(seed)
    return {
        "hard": C.onehot(lab, K),
        "soft": P.astype("float32"),
        "hard+soft": np.hstack([C.onehot(lab, K), P]).astype("float32"),
        "cos_sim": cs.astype("float32"),
        "emb_pca16": Z_pca_a.astype("float32"),
        "emb_vol_orth": Z_orth_pca_a.astype("float32"),
        "null_random": C.onehot(rng.integers(0, K, Z_a.shape[0]), K)
    }

cv_all, test_all = [], []
for K in DOWNSTREAM_KS:
    mk = models[K]
    var_tr = make_variants(mk, K, Z_tr, Z_tr_pca, Z_tr_orth_pca, seed=0)
    var_te = make_variants(mk, K, Z_te, Z_te_pca, Z_te_orth_pca, seed=1)
    summ, wide = C.downstream_cv(A["base_train"], y_tr, var_tr, CV_SAMPLE, CV_SPLITS, CV_REPEATS)
    summ.insert(0, "K", K); summ.insert(0, "method", METHOD); cv_all.append(summ)
    print(f"\n=== {METHOD} Train CV, K={K} ==="); display(summ)
    ft = C.final_test_eval(A["base_train"], y_tr, A["base_test"], y_te, var_tr, var_te)
    ft.insert(0, "K", K); ft.insert(0, "method", METHOD); test_all.append(ft)
    print(f"=== {METHOD} Real TEST Set, K={K} ==="); display(ft)

pd.concat(cv_all).to_csv(f"{WORK}/results/{METHOD}_downstream_cv.csv")
pd.concat(test_all).to_csv(f"{WORK}/results/{METHOD}_downstream_test.csv")



=== gru_cl Train CV, K=5 ===


,method,K,auc_mean,auc_std,delta_vs_base,delta_std,folds_better
variant,,,,,,,
baseline,gru_cl,5,0.85318,0.00205,0.00000,0.00000,0
hard,gru_cl,5,0.85331,0.00207,0.00013,0.00045,6
soft,gru_cl,5,0.85282,0.00165,-0.00036,0.00079,4
hard+soft,gru_cl,5,0.85279,0.00169,-0.00039,0.00067,4
cos_sim,gru_cl,5,0.85296,0.00234,-0.00022,0.00063,3
emb_pca16,gru_cl,5,0.85260,0.00240,-0.00058,0.00093,4
emb_vol_orth,gru_cl,5,0.85243,0.00220,-0.00075,0.00058,0
null_random,gru_cl,5,0.85312,0.00221,-0.00007,0.00038,3


=== gru_cl Real TEST Set, K=5 ===


method  K      auc                ap                f1           \
                            mean      std     mean      std     mean      std   
variant                                                                         
baseline      gru_cl  5  0.85305  0.00006  0.94705  0.00008  0.91002  0.00028   
hard          gru_cl  5  0.85310  0.00011  0.94711  0.00021  0.90974  0.00041   
soft          gru_cl  5  0.85286  0.00018  0.94698  0.00004  0.90937  0.00046   
hard+soft     gru_cl  5  0.85286  0.00015  0.94699  0.00009  0.90934  0.00063   
cos_sim       gru_cl  5  0.85282  0.00025  0.94702  0.00019  0.90984  0.00011   
emb_pca16     gru_cl  5  0.85284  0.00010  0.94691  0.00012  0.90952  0.00038   
emb_vol_orth  gru_cl  5  0.85284  0.00007  0.94703  0.00009  0.90989  0.00022   
null_random   gru_cl  5  0.85309  0.00005  0.94706  0.00012  0.90989  0.00020   

             auc_delta_vs_base  
                                
variant                         
baseline               0.00000  
hard                   0.00005  
soft                  -0.00019  
hard+soft             -0.00019  
cos_sim               -0.00023  
emb_pca16             -0.00021  
emb_vol_orth          -0.00021  
null_random            0.00004


=== gru_cl Train CV, K=8 ===


,method,K,auc_mean,auc_std,delta_vs_base,delta_std,folds_better
variant,,,,,,,
baseline,gru_cl,8,0.85318,0.00205,0.00000,0.00000,0
hard,gru_cl,8,0.85307,0.00224,-0.00011,0.00046,2
soft,gru_cl,8,0.85277,0.00192,-0.00041,0.00048,2
hard+soft,gru_cl,8,0.85260,0.00202,-0.00058,0.00041,1
cos_sim,gru_cl,8,0.85277,0.00209,-0.00042,0.00059,3
emb_pca16,gru_cl,8,0.85260,0.00240,-0.00058,0.00093,4
emb_vol_orth,gru_cl,8,0.85243,0.00220,-0.00075,0.00058,0
null_random,gru_cl,8,0.85319,0.00190,0.00001,0.00051,5


=== gru_cl Real TEST Set, K=8 ===


method  K      auc                ap                f1           \
                            mean      std     mean      std     mean      std   
variant                                                                         
baseline      gru_cl  8  0.85305  0.00006  0.94705  0.00008  0.91002  0.00028   
hard          gru_cl  8  0.85302  0.00016  0.94708  0.00013  0.91004  0.00036   
soft          gru_cl  8  0.85271  0.00028  0.94680  0.00013  0.90975  0.00057   
hard+soft     gru_cl  8  0.85277  0.00025  0.94698  0.00011  0.90983  0.00006   
cos_sim       gru_cl  8  0.85281  0.00009  0.94696  0.00013  0.90967  0.00003   
emb_pca16     gru_cl  8  0.85284  0.00010  0.94691  0.00012  0.90952  0.00038   
emb_vol_orth  gru_cl  8  0.85284  0.00007  0.94703  0.00009  0.90989  0.00022   
null_random   gru_cl  8  0.85301  0.00006  0.94700  0.00001  0.90986  0.00060   

             auc_delta_vs_base  
                                
variant                         
baseline               0.00000  
hard                  -0.00003  
soft                  -0.00034  
hard+soft             -0.00028  
cos_sim               -0.00024  
emb_pca16             -0.00021  
emb_vol_orth          -0.00021  
null_random           -0.00004

## 7. Save Artifacts & Summary JSON

In [9]:
np.save(f"{WORK}/results/feat_{METHOD}_train_emb.npy", Z_tr)
np.save(f"{WORK}/results/feat_{METHOD}_test_emb.npy",  Z_te)
np.save(f"{WORK}/results/feat_{METHOD}_train_emb_orth.npy", Z_tr_orth)
np.save(f"{WORK}/results/feat_{METHOD}_test_emb_orth.npy",  Z_te_orth)
torch.save(model.state_dict(), f"{WORK}/results/{METHOD}_weights.pt")

for K_save in DOWNSTREAM_KS:
    mk = models[K_save]
    for s, ZZ in (("train", Z_tr), ("test", Z_te)):
        P = mk.predict_proba(ZZ)
        np.save(f"{WORK}/results/feat_{METHOD}_K{K_save}_{s}_soft.npy", P.astype("float32"))
        np.save(f"{WORK}/results/feat_{METHOD}_K{K_save}_{s}_cossim.npy", mk.cosine_sim(ZZ).astype("float32"))
        pd.DataFrame(P, index=A[f"ids_{s}"], columns=[f"p{k}" for k in range(K_save)]).assign(cluster=P.argmax(1)) \
          .to_csv(f"{WORK}/results/{METHOD}_labels_K{K_save}_{s}.csv", index_label="enrollment_id")

summary = dict(method=METHOD, K=K_FINAL, max_len=MAX_LEN, val_loss=float(best_val_loss))
summary.update({k: float(v) for k, v in grid.loc[K_FINAL].items()})
summary.update(stab_by_k[K_FINAL])
summary["K5_summary"] = {
    **{k: float(v) for k, v in grid.loc[5].items()},
    **stab_by_k[5]
}
json.dump(summary, open(f"{WORK}/results/summary_{METHOD}.json", "w"), indent=1)
print(json.dumps(summary, indent=1))


{
 "method": "gru_cl",
 "K": 8,
 "max_len": 128,
 "val_loss": 2.4331866746322586,
 "inertia": 28979.072265625,
 "mean_max_posterior": 0.7670649886131287,
 "n_clusters": 8.0,
 "min_cluster_share": 0.058485838960694196,
 "silhouette": 0.18990732729434967,
 "davies_bouldin": 5.7590824585451506,
 "calinski_harabasz": 1142.263671875,
 "native_silhouette": 0.38483887910842896,
 "native_davies_bouldin": 1.4254652465443771,
 "native_calinski_harabasz": 8134.6513671875,
 "ari_vs_volume": 0.4987069903794962,
 "nmi_vs_volume": 0.6295058372296929,
 "eta2_logvolume": 0.9423080389194485,
 "cramers_v_train": 0.4983,
 "cramers_v_test": 0.4916,
 "seed_ari_mean": 0.8719,
 "seed_ari_min": 0.7771,
 "boot_ari_mean": 0.8474,
 "boot_ari_std": 0.1201,
 "share_L1_train_vs_test": 0.0077,
 "dropout_corr_train_vs_test": 0.9999,
 "K5_summary": {
  "inertia": 36469.79296875,
  "mean_max_posterior": 0.886660635471344,
  "n_clusters": 5.0,
  "min_cluster_share": 0.16928539430240083,
  "silhouette": 0.1877965033054351